In [ ]:
import lightkurve as lk
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
%matplotlib inline

In [ ]:
lk.conf.cache_dir = './'  # 设置 Lightkurve 的缓存目录

In [ ]:
kic = 'KIC001296068'

In [ ]:
lc_search = lk.search_lightcurve(f'{kic}', author='Kepler',cadence='long')
lc_search

In [ ]:
lc = lc_search.download_all()

In [ ]:
lc
lc.plot()

In [ ]:
lc_1 = lc.stitch()
lc_1.plot()

In [ ]:
lc_a = lc
for i in range(len(lc_a)):
    lc_a[i] = lc_a[i].flatten(window_length=201)
lc_a = lc_a.stitch()
lc_a.plot()

In [ ]:
pg = lc_a.normalize(unit='ppm').to_periodogram().flatten()

In [ ]:
numax = 59.560
dnu = 6.140
lower_freq = numax - 3 * dnu
upper_freq = numax + 3 * dnu

frequency_uHz = pg.frequency.value * 1e6 / (24 * 3600)  # 1/day 转换为 uHz
power = pg.power.value

mask = (frequency_uHz >= lower_freq) & (frequency_uHz <= upper_freq)
frequency_selected = frequency_uHz[mask]
power_selected = power[mask]

plt.figure(figsize=(10, 6))
plt.plot(frequency_selected, power_selected, color='blue')
plt.axvline(x=numax, color='red', linestyle='--', linewidth=1.5, label=r'$\nu_{\mathrm{max}}$')
y_pos = power_selected.max() * 0.95  # 在功率谱的95%位置添加文本
plt.text(numax, y_pos, f'numax = {numax:.2f} μHz', color='red', fontsize=12,
            ha='left', va='top', backgroundcolor='white')
plt.xlabel("Frequency (uHz)", fontsize=14)
plt.ylabel("Power (ppm)", fontsize=14)
plt.title(f"{kic} Power Spectrum\nFrequency Range: {lower_freq:.2f} - {upper_freq:.2f} uHz", fontsize=16)
plt.grid(True, which='both', ls='--', lw=0.5)
plt.tight_layout()
plt.show()